# OCR-система для распознавания текста на фотографиях городской среды

Вариант 25. Text detection + text recognition.

## 1. Проверка структуры датасета

In [ ]:
from pathlib import Path
import os

# Определяем папку проекта
if Path.cwd().name.lower() == "notebooks":
    PROJECT_DIR = Path.cwd().parent
else:
    PROJECT_DIR = Path.cwd()

DATASET_DIR = PROJECT_DIR / "data" / "raw" / "coco_text_v2" / "archive"
IMAGES_DIR = DATASET_DIR / "data"

print("Папка проекта:", PROJECT_DIR)
print("Папка датасета:", DATASET_DIR)
print("Папка с изображениями и разметкой:", IMAGES_DIR)

print("\nПроверка существования:")
print("DATASET_DIR exists:", DATASET_DIR.exists())
print("IMAGES_DIR exists:", IMAGES_DIR.exists())
print("train.txt exists:", (DATASET_DIR / "train.txt").exists())
print("val.txt exists:", (DATASET_DIR / "val.txt").exists())

jpg_files = list(IMAGES_DIR.glob("*.jpg"))
txt_files = list(IMAGES_DIR.glob("*.txt"))

print("\nКоличество файлов:")
print("Изображений .jpg:", len(jpg_files))
print("Файлов разметки .txt:", len(txt_files))

print("\nПервые 5 изображений:")
for file in jpg_files[:5]:
    print(file.name)

print("\nПример содержимого разметки 0.txt:")
with open(IMAGES_DIR / "0.txt", "r", encoding="utf-8") as f:
    print(f.read())

## 2. Создание конфигурационного файла для text detection

In [ ]:
from pathlib import Path

if Path.cwd().name.lower() == "notebooks":
    PROJECT_DIR = Path.cwd().parent
else:
    PROJECT_DIR = Path.cwd()

DATASET_DIR = PROJECT_DIR / "data" / "raw" / "coco_text_v2" / "archive"
YAML_PATH = PROJECT_DIR / "ocr_text_detection.yaml"

yaml_content = f"""
path: {DATASET_DIR.as_posix()}
train: train.txt
val: val.txt

nc: 1
names: ['text']
"""

with open(YAML_PATH, "w", encoding="utf-8") as f:
    f.write(yaml_content)

print("Конфиг создан:", YAML_PATH)
print("\nСодержимое конфига:")
print(yaml_content)

## 3. Проверка рабочей среды и библиотек

In [ ]:
import cv2
import pandas as pd
import numpy as np
from ultralytics import YOLO
import easyocr

print("OpenCV:", cv2.__version__)
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Ultralytics OK")
print("EasyOCR OK")


## 4. Визуальная проверка разметки

In [ ]:
from pathlib import Path
import cv2
import matplotlib.pyplot as plt
import random

# Определяем пути
if Path.cwd().name.lower() == "notebooks":
    PROJECT_DIR = Path.cwd().parent
else:
    PROJECT_DIR = Path.cwd()

DATASET_DIR = PROJECT_DIR / "data" / "raw" / "coco_text_v2" / "archive"
IMAGES_DIR = DATASET_DIR / "data"
REPORT_DIR = PROJECT_DIR / "report_materials"
REPORT_DIR.mkdir(exist_ok=True)

# Берём несколько случайных изображений
image_files = list(IMAGES_DIR.glob("*.jpg"))
sample_images = random.sample(image_files, 3)

for img_path in sample_images:
    label_path = img_path.with_suffix(".txt")
    
    image = cv2.imread(str(img_path))
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    h, w = image.shape[:2]
    
    if label_path.exists():
        with open(label_path, "r", encoding="utf-8") as f:
            lines = f.readlines()
        
        for line in lines:
            parts = line.strip().split()
            if len(parts) != 5:
                continue
            
            class_id, x_center, y_center, box_width, box_height = map(float, parts)
            
            # YOLO -> обычные координаты
            x_center *= w
            y_center *= h
            box_width *= w
            box_height *= h
            
            x1 = int(x_center - box_width / 2)
            y1 = int(y_center - box_height / 2)
            x2 = int(x_center + box_width / 2)
            y2 = int(y_center + box_height / 2)
            
            cv2.rectangle(image, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(image, "text", (x1, max(y1 - 5, 15)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
    
    plt.figure(figsize=(10, 6))
    plt.imshow(image)
    plt.title(f"Проверка разметки: {img_path.name}")
    plt.axis("off")
    plt.show()

## 5. Проверка CUDA и доступности GPU

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA доступна:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU не используется, обучение будет идти на CPU")

## 6. Подготовка train.txt и val.txt для локального запуска

In [ ]:
from pathlib import Path

if Path.cwd().name.lower() == "notebooks":
    PROJECT_DIR = Path.cwd().parent
else:
    PROJECT_DIR = Path.cwd()

DATASET_DIR = PROJECT_DIR / "data" / "raw" / "coco_text_v2" / "archive"
IMAGES_DIR = DATASET_DIR / "data"

print("DATASET_DIR:", DATASET_DIR)
print("IMAGES_DIR:", IMAGES_DIR)

for split_file in ["train.txt", "val.txt"]:
    split_path = DATASET_DIR / split_file
    backup_path = DATASET_DIR / f"{split_file}.backup"

    # Сохраняем резервную копию исходного файла
    if not backup_path.exists():
        backup_path.write_text(split_path.read_text(encoding="utf-8"), encoding="utf-8")
        print(f"Создана резервная копия: {backup_path.name}")

    # Читаем старые строки и берём только имя изображения
    old_lines = split_path.read_text(encoding="utf-8").splitlines()

    new_lines = []
    for line in old_lines:
        if not line.strip():
            continue

        img_name = Path(line.strip()).name
        img_path = IMAGES_DIR / img_name

        # Пишем абсолютный путь, чтобы YOLO точно нашёл файл
        new_lines.append(img_path.as_posix())

    split_path.write_text("\n".join(new_lines), encoding="utf-8")
    print(f"{split_file} переписан, строк: {len(new_lines)}")

print("\nПервые строки после исправления:")
for split_file in ["train.txt", "val.txt"]:
    split_path = DATASET_DIR / split_file
    print("\n", split_file)
    lines = split_path.read_text(encoding="utf-8").splitlines()
    for line in lines[:5]:
        img_path = Path(line)
        label_path = img_path.with_suffix(".txt")
        print(line)
        print("  image exists:", img_path.exists(), "| label exists:", label_path.exists())

## 7. Очистка кэша Ultralytics

In [ ]:
from pathlib import Path

for cache_file in DATASET_DIR.rglob("*.cache"):
    print("Удаляю cache:", cache_file)
    cache_file.unlink()

print("Кэш YOLO очищен")

## 8. Обучение и оценка YOLOv8n

In [ ]:
from ultralytics import YOLO
from pathlib import Path

if Path.cwd().name.lower() == "notebooks":
    PROJECT_DIR = Path.cwd().parent
else:
    PROJECT_DIR = Path.cwd()

YAML_PATH = PROJECT_DIR / "ocr_text_detection.yaml"

model = YOLO("yolov8n.pt")

results = model.train(
    data=str(YAML_PATH),
    epochs=10,
    imgsz=640,
    batch=16,
    device=0,
    project=str(PROJECT_DIR / "models"),
    name="yolov8n_text_detection",
    workers=0
)

In [ ]:
from pathlib import Path

PROJECT_DIR = Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd()

RUN_DIR = PROJECT_DIR / "models" / "yolov8n_text_detection-2"
WEIGHTS_PATH = RUN_DIR / "weights" / "best.pt"
YAML_PATH = PROJECT_DIR / "ocr_text_detection.yaml"

print("Папка проекта:", PROJECT_DIR)
print("Папка обучения:", RUN_DIR)
print("Файл весов:", WEIGHTS_PATH)
print("best.pt существует:", WEIGHTS_PATH.exists())
print("YAML существует:", YAML_PATH.exists())

In [ ]:
from ultralytics import YOLO
import json
from pathlib import Path

model = YOLO(str(WEIGHTS_PATH))

metrics = model.val(
    data=str(YAML_PATH),
    imgsz=640,
    batch=16,
    device=0,
    workers=0
)

yolov8n_results = {
    "model": "YOLOv8n",
    "task": "text detection",
    "training_scheme": "fine-tuning",
    "epochs": 10,
    "imgsz": 640,
    "mAP50": float(metrics.box.map50),
    "mAP50_95": float(metrics.box.map),
    "precision": float(metrics.box.mp),
    "recall": float(metrics.box.mr),
    "speed_ms_preprocess": float(metrics.speed["preprocess"]),
    "speed_ms_inference": float(metrics.speed["inference"]),
    "speed_ms_postprocess": float(metrics.speed["postprocess"]),
}

RESULTS_JSON_DIR = PROJECT_DIR / "results" / "json"
RESULTS_JSON_DIR.mkdir(parents=True, exist_ok=True)

with open(RESULTS_JSON_DIR / "yolov8n_metrics.json", "w", encoding="utf-8") as f:
    json.dump(yolov8n_results, f, ensure_ascii=False, indent=4)

print(json.dumps(yolov8n_results, ensure_ascii=False, indent=4))
print("\nМетрики сохранены:", RESULTS_JSON_DIR / "yolov8n_metrics.json")

In [ ]:
model_size_mb = WEIGHTS_PATH.stat().st_size / (1024 * 1024)

print(f"Размер модели YOLOv8n best.pt: {model_size_mb:.2f} MB")

## 9. Обучение и оценка YOLO11n

In [ ]:
from ultralytics import YOLO
from pathlib import Path

PROJECT_DIR = Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd()
YAML_PATH = PROJECT_DIR / "ocr_text_detection.yaml"

model = YOLO("yolo11n.pt")

results = model.train(
    data=str(YAML_PATH),
    epochs=10,
    imgsz=640,
    batch=16,
    device=0,
    project=str(PROJECT_DIR / "models"),
    name="yolo11n_text_detection",
    workers=0
)

In [ ]:
from pathlib import Path

PROJECT_DIR = Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd()

RUN_DIR_YOLO11 = PROJECT_DIR / "models" / "yolo11n_text_detection"
WEIGHTS_PATH_YOLO11 = RUN_DIR_YOLO11 / "weights" / "best.pt"
YAML_PATH = PROJECT_DIR / "ocr_text_detection.yaml"

print("Папка обучения YOLO11n:", RUN_DIR_YOLO11)
print("best.pt существует:", WEIGHTS_PATH_YOLO11.exists())
print("last.pt существует:", (RUN_DIR_YOLO11 / "weights" / "last.pt").exists())
print("YAML существует:", YAML_PATH.exists())

In [ ]:
from ultralytics import YOLO
import json
from pathlib import Path

model = YOLO(str(WEIGHTS_PATH_YOLO11))

metrics = model.val(
    data=str(YAML_PATH),
    imgsz=640,
    batch=16,
    device=0,
    workers=0
)

yolo11n_results = {
    "model": "YOLO11n",
    "task": "text detection",
    "training_scheme": "fine-tuning",
    "epochs": 10,
    "imgsz": 640,
    "mAP50": float(metrics.box.map50),
    "mAP50_95": float(metrics.box.map),
    "precision": float(metrics.box.mp),
    "recall": float(metrics.box.mr),
    "speed_ms_preprocess": float(metrics.speed["preprocess"]),
    "speed_ms_inference": float(metrics.speed["inference"]),
    "speed_ms_postprocess": float(metrics.speed["postprocess"]),
}

RESULTS_JSON_DIR = PROJECT_DIR / "results" / "json"
RESULTS_JSON_DIR.mkdir(parents=True, exist_ok=True)

with open(RESULTS_JSON_DIR / "yolo11n_metrics.json", "w", encoding="utf-8") as f:
    json.dump(yolo11n_results, f, ensure_ascii=False, indent=4)

print(json.dumps(yolo11n_results, ensure_ascii=False, indent=4))
print("\nМетрики сохранены:", RESULTS_JSON_DIR / "yolo11n_metrics.json")

In [ ]:
model_size_mb_yolo11 = WEIGHTS_PATH_YOLO11.stat().st_size / (1024 * 1024)

print(f"Размер модели YOLO11n best.pt: {model_size_mb_yolo11:.2f} MB")

## 10. Обучение и оценка RT-DETR-L

In [ ]:
from ultralytics import YOLO
from pathlib import Path

PROJECT_DIR = Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd()
YAML_PATH = PROJECT_DIR / "ocr_text_detection.yaml"

model = YOLO("rtdetr-l.pt")

results = model.train(
    data=str(YAML_PATH),
    epochs=10,
    imgsz=640,
    batch=4,
    device=0,
    project=str(PROJECT_DIR / "models"),
    name="rtdetr_text_detection",
    workers=0
)

In [ ]:
from pathlib import Path

PROJECT_DIR = Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd()

RUN_DIR_RTDETR = PROJECT_DIR / "models" / "rtdetr_text_detection"
WEIGHTS_PATH_RTDETR = RUN_DIR_RTDETR / "weights" / "best.pt"
YAML_PATH = PROJECT_DIR / "ocr_text_detection.yaml"

print("Папка обучения RT-DETR:", RUN_DIR_RTDETR)
print("best.pt существует:", WEIGHTS_PATH_RTDETR.exists())
print("last.pt существует:", (RUN_DIR_RTDETR / "weights" / "last.pt").exists())
print("YAML существует:", YAML_PATH.exists())

In [ ]:
from ultralytics import YOLO
import json
from pathlib import Path

model = YOLO(str(WEIGHTS_PATH_RTDETR))

metrics = model.val(
    data=str(YAML_PATH),
    imgsz=640,
    batch=4,
    device=0,
    workers=0
)

rtdetr_results = {
    "model": "RT-DETR-L",
    "task": "text detection",
    "training_scheme": "fine-tuning",
    "epochs": 10,
    "imgsz": 640,
    "mAP50": float(metrics.box.map50),
    "mAP50_95": float(metrics.box.map),
    "precision": float(metrics.box.mp),
    "recall": float(metrics.box.mr),
    "speed_ms_preprocess": float(metrics.speed["preprocess"]),
    "speed_ms_inference": float(metrics.speed["inference"]),
    "speed_ms_postprocess": float(metrics.speed["postprocess"]),
}

RESULTS_JSON_DIR = PROJECT_DIR / "results" / "json"
RESULTS_JSON_DIR.mkdir(parents=True, exist_ok=True)

with open(RESULTS_JSON_DIR / "rtdetr_metrics.json", "w", encoding="utf-8") as f:
    json.dump(rtdetr_results, f, ensure_ascii=False, indent=4)

print(json.dumps(rtdetr_results, ensure_ascii=False, indent=4))
print("\nМетрики сохранены:", RESULTS_JSON_DIR / "rtdetr_metrics.json")

In [ ]:
model_size_mb_rtdetr = WEIGHTS_PATH_RTDETR.stat().st_size / (1024 * 1024)

print(f"Размер модели RT-DETR-L best.pt: {model_size_mb_rtdetr:.2f} MB")

## 11. Обучение и оценка YOLOv10n

In [ ]:
from ultralytics import YOLO
from pathlib import Path

PROJECT_DIR = Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd()
YAML_PATH = PROJECT_DIR / "ocr_text_detection.yaml"

model = YOLO("yolov10n.pt")

results = model.train(
    data=str(YAML_PATH),
    epochs=10,
    imgsz=640,
    batch=16,
    device=0,
    project=str(PROJECT_DIR / "models"),
    name="yolov10n_text_detection",
    workers=0
)

In [ ]:
from pathlib import Path

PROJECT_DIR = Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd()

RUN_DIR_YOLO10 = PROJECT_DIR / "models" / "yolov10n_text_detection"
WEIGHTS_PATH_YOLO10 = RUN_DIR_YOLO10 / "weights" / "best.pt"
YAML_PATH = PROJECT_DIR / "ocr_text_detection.yaml"

print("Папка обучения YOLOv10n:", RUN_DIR_YOLO10)
print("best.pt существует:", WEIGHTS_PATH_YOLO10.exists())
print("last.pt существует:", (RUN_DIR_YOLO10 / "weights" / "last.pt").exists())
print("YAML существует:", YAML_PATH.exists())

In [ ]:
from ultralytics import YOLO
import json
from pathlib import Path

model = YOLO(str(WEIGHTS_PATH_YOLO10))

metrics = model.val(
    data=str(YAML_PATH),
    imgsz=640,
    batch=16,
    device=0,
    workers=0
)

yolov10n_results = {
    "model": "YOLOv10n",
    "task": "text detection",
    "training_scheme": "fine-tuning",
    "epochs": 10,
    "imgsz": 640,
    "mAP50": float(metrics.box.map50),
    "mAP50_95": float(metrics.box.map),
    "precision": float(metrics.box.mp),
    "recall": float(metrics.box.mr),
    "speed_ms_preprocess": float(metrics.speed["preprocess"]),
    "speed_ms_inference": float(metrics.speed["inference"]),
    "speed_ms_postprocess": float(metrics.speed["postprocess"]),
}

RESULTS_JSON_DIR = PROJECT_DIR / "results" / "json"
RESULTS_JSON_DIR.mkdir(parents=True, exist_ok=True)

with open(RESULTS_JSON_DIR / "yolov10n_metrics.json", "w", encoding="utf-8") as f:
    json.dump(yolov10n_results, f, ensure_ascii=False, indent=4)

print(json.dumps(yolov10n_results, ensure_ascii=False, indent=4))
print("\nМетрики сохранены:", RESULTS_JSON_DIR / "yolov10n_metrics.json")

In [ ]:
model_size_mb_yolo10 = WEIGHTS_PATH_YOLO10.stat().st_size / (1024 * 1024)

print(f"Размер модели YOLOv10n best.pt: {model_size_mb_yolo10:.2f} MB")

## 12. Обучение и оценка Faster R-CNN ResNet50-FPN

In [ ]:
from pathlib import Path
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as T

PROJECT_DIR = Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd()
DATASET_DIR = PROJECT_DIR / "data" / "raw" / "coco_text_v2" / "archive"
IMAGES_DIR = DATASET_DIR / "data"

class TextDetectionDataset(Dataset):
    def __init__(self, split_file, transforms=None, limit=None):
        self.transforms = transforms
        self.image_paths = []
        
        with open(split_file, "r", encoding="utf-8") as f:
            lines = f.read().splitlines()
        
        for line in lines:
            if line.strip():
                self.image_paths.append(Path(line.strip()))
        
        if limit is not None:
            self.image_paths = self.image_paths[:limit]
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        label_path = img_path.with_suffix(".txt")
        
        image = Image.open(img_path).convert("RGB")
        w, h = image.size
        
        boxes = []
        labels = []
        
        if label_path.exists():
            with open(label_path, "r", encoding="utf-8") as f:
                lines = f.readlines()
            
            for line in lines:
                parts = line.strip().split()
                if len(parts) != 5:
                    continue
                
                class_id, x_center, y_center, box_width, box_height = map(float, parts)
                
                x_center *= w
                y_center *= h
                box_width *= w
                box_height *= h
                
                x1 = x_center - box_width / 2
                y1 = y_center - box_height / 2
                x2 = x_center + box_width / 2
                y2 = y_center + box_height / 2
                
                if x2 > x1 and y2 > y1:
                    boxes.append([x1, y1, x2, y2])
                    labels.append(1)
        
        if len(boxes) == 0:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)
        else:
            boxes = torch.as_tensor(boxes, dtype=torch.float32)
            labels = torch.as_tensor(labels, dtype=torch.int64)
        
        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": torch.tensor([idx])
        }
        
        if self.transforms:
            image = self.transforms(image)
        
        return image, target

def collate_fn(batch):
    return tuple(zip(*batch))

transform = T.ToTensor()

train_dataset_frcnn = TextDetectionDataset(
    DATASET_DIR / "train.txt",
    transforms=transform,
    limit=3000
)

val_dataset_frcnn = TextDetectionDataset(
    DATASET_DIR / "val.txt",
    transforms=transform,
    limit=800
)

train_loader_frcnn = DataLoader(
    train_dataset_frcnn,
    batch_size=2,
    shuffle=True,
    collate_fn=collate_fn
)

val_loader_frcnn = DataLoader(
    val_dataset_frcnn,
    batch_size=2,
    shuffle=False,
    collate_fn=collate_fn
)

print("Train Faster R-CNN images:", len(train_dataset_frcnn))
print("Val Faster R-CNN images:", len(val_dataset_frcnn))

In [ ]:
import torch
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from pathlib import Path
import time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_frcnn = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights="DEFAULT")

num_classes = 2  # background + text
in_features = model_frcnn.roi_heads.box_predictor.cls_score.in_features
model_frcnn.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

model_frcnn.to(device)

params = [p for p in model_frcnn.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(
    params,
    lr=0.005,
    momentum=0.9,
    weight_decay=0.0005
)

num_epochs = 3

PROJECT_DIR = Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd()
FRCNN_DIR = PROJECT_DIR / "models" / "faster_rcnn_text_detection"
FRCNN_DIR.mkdir(parents=True, exist_ok=True)

print("Start Faster R-CNN training")
print("Device:", device)

for epoch in range(num_epochs):
    model_frcnn.train()
    epoch_loss = 0
    start_time = time.time()
    
    for batch_idx, (images, targets) in enumerate(train_loader_frcnn):
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        
        loss_dict = model_frcnn(images, targets)
        losses = sum(loss for loss in loss_dict.values())
        
        optimizer.zero_grad()
        losses.backward()
        optimizer.step()
        
        epoch_loss += losses.item()
        
        if batch_idx % 100 == 0:
            print(
                f"Epoch [{epoch+1}/{num_epochs}], "
                f"Batch [{batch_idx}/{len(train_loader_frcnn)}], "
                f"Loss: {losses.item():.4f}"
            )
    
    epoch_time = time.time() - start_time
    avg_loss = epoch_loss / len(train_loader_frcnn)
    print(
        f"Epoch [{epoch+1}/{num_epochs}] finished. "
        f"Avg loss: {avg_loss:.4f}, Time: {epoch_time:.1f} sec"
    )

torch.save(model_frcnn.state_dict(), FRCNN_DIR / "faster_rcnn_text_detection.pth")

print("Faster R-CNN model saved:", FRCNN_DIR / "faster_rcnn_text_detection.pth")

In [ ]:
import torch
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from pathlib import Path
import time
import json
import numpy as np

PROJECT_DIR = Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd()
FRCNN_DIR = PROJECT_DIR / "models" / "faster_rcnn_text_detection"
FRCNN_WEIGHTS_PATH = FRCNN_DIR / "faster_rcnn_text_detection.pth"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Воссоздаём архитектуру Faster R-CNN
model_frcnn_eval = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights=None)

num_classes = 2  # background + text
in_features = model_frcnn_eval.roi_heads.box_predictor.cls_score.in_features
model_frcnn_eval.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

# Загружаем обученные веса
model_frcnn_eval.load_state_dict(torch.load(FRCNN_WEIGHTS_PATH, map_location=device))
model_frcnn_eval.to(device)
model_frcnn_eval.eval()

print("Faster R-CNN weights loaded:", FRCNN_WEIGHTS_PATH)
print("Device:", device)

In [ ]:
def box_iou(box1, box2):
    """
    box1: tensor [N, 4]
    box2: tensor [M, 4]
    """
    if box1.numel() == 0 or box2.numel() == 0:
        return torch.zeros((box1.shape[0], box2.shape[0]))

    area1 = (box1[:, 2] - box1[:, 0]).clamp(0) * (box1[:, 3] - box1[:, 1]).clamp(0)
    area2 = (box2[:, 2] - box2[:, 0]).clamp(0) * (box2[:, 3] - box2[:, 1]).clamp(0)

    lt = torch.max(box1[:, None, :2], box2[:, :2])
    rb = torch.min(box1[:, None, 2:], box2[:, 2:])

    wh = (rb - lt).clamp(min=0)
    inter = wh[:, :, 0] * wh[:, :, 1]

    union = area1[:, None] + area2 - inter
    return inter / union.clamp(min=1e-6)


def compute_ap(recall, precision):
    recall = np.concatenate(([0.0], recall, [1.0]))
    precision = np.concatenate(([0.0], precision, [0.0]))

    for i in range(len(precision) - 1, 0, -1):
        precision[i - 1] = max(precision[i - 1], precision[i])

    indices = np.where(recall[1:] != recall[:-1])[0]
    ap = np.sum((recall[indices + 1] - recall[indices]) * precision[indices + 1])
    return ap


def evaluate_faster_rcnn(model, data_loader, device, iou_thresholds=None, score_threshold=0.25):
    if iou_thresholds is None:
        iou_thresholds = np.arange(0.5, 1.0, 0.05)

    model.eval()

    all_predictions = []
    all_ground_truths = {}
    image_counter = 0

    total_inference_time = 0.0
    total_images = 0

    with torch.no_grad():
        for images, targets in data_loader:
            images_gpu = [img.to(device) for img in images]

            start_time = time.time()
            outputs = model(images_gpu)
            if device.type == "cuda":
                torch.cuda.synchronize()
            end_time = time.time()

            total_inference_time += end_time - start_time
            total_images += len(images)

            for output, target in zip(outputs, targets):
                image_id = image_counter

                gt_boxes = target["boxes"].cpu()
                all_ground_truths[image_id] = {
                    "boxes": gt_boxes,
                    "matched": {}
                }

                pred_boxes = output["boxes"].cpu()
                pred_scores = output["scores"].cpu()

                for box, score in zip(pred_boxes, pred_scores):
                    all_predictions.append({
                        "image_id": image_id,
                        "box": box,
                        "score": float(score)
                    })

                image_counter += 1

    # Сортируем предсказания по уверенности
    all_predictions = sorted(all_predictions, key=lambda x: x["score"], reverse=True)

    total_gt = sum(len(item["boxes"]) for item in all_ground_truths.values())

    ap_values = []
    precision_at_50 = 0.0
    recall_at_50 = 0.0
    false_positives_at_50 = 0
    missed_at_50 = 0

    for iou_thr in iou_thresholds:
        tp = []
        fp = []

        for gt in all_ground_truths.values():
            gt["matched"][iou_thr] = set()

        for pred in all_predictions:
            image_id = pred["image_id"]
            pred_box = pred["box"].unsqueeze(0)
            gt_boxes = all_ground_truths[image_id]["boxes"]

            if len(gt_boxes) == 0:
                tp.append(0)
                fp.append(1)
                continue

            ious = box_iou(pred_box, gt_boxes).squeeze(0)
            max_iou, max_idx = torch.max(ious, dim=0)

            if max_iou >= iou_thr and int(max_idx) not in all_ground_truths[image_id]["matched"][iou_thr]:
                tp.append(1)
                fp.append(0)
                all_ground_truths[image_id]["matched"][iou_thr].add(int(max_idx))
            else:
                tp.append(0)
                fp.append(1)

        tp = np.array(tp)
        fp = np.array(fp)

        if len(tp) == 0:
            ap = 0.0
            precision = 0.0
            recall = 0.0
        else:
            tp_cumsum = np.cumsum(tp)
            fp_cumsum = np.cumsum(fp)

            recall_curve = tp_cumsum / max(total_gt, 1)
            precision_curve = tp_cumsum / np.maximum(tp_cumsum + fp_cumsum, 1)

            ap = compute_ap(recall_curve, precision_curve)
            precision = precision_curve[-1]
            recall = recall_curve[-1]

        ap_values.append(ap)

        if abs(iou_thr - 0.5) < 1e-6:
            precision_at_50 = precision
            recall_at_50 = recall
            false_positives_at_50 = int(fp.sum()) if len(fp) > 0 else 0
            missed_at_50 = int(total_gt - tp.sum()) if len(tp) > 0 else total_gt

    avg_inference_ms = (total_inference_time / max(total_images, 1)) * 1000

    return {
        "mAP50": float(ap_values[0]),
        "mAP50_95": float(np.mean(ap_values)),
        "precision": float(precision_at_50),
        "recall": float(recall_at_50),
        "false_positives": false_positives_at_50,
        "missed_objects": missed_at_50,
        "speed_ms_inference": float(avg_inference_ms),
        "evaluated_images": total_images,
        "ground_truth_objects": total_gt
    }


frcnn_metrics = evaluate_faster_rcnn(
    model_frcnn_eval,
    val_loader_frcnn,
    device,
    score_threshold=0.25
)

frcnn_results = {
    "model": "Faster R-CNN ResNet50-FPN",
    "task": "text detection",
    "training_scheme": "fine-tuning",
    "epochs": 3,
    "imgsz": "original",
    "train_subset": 3000,
    "val_subset": 800,
    **frcnn_metrics
}

RESULTS_JSON_DIR = PROJECT_DIR / "results" / "json"
RESULTS_JSON_DIR.mkdir(parents=True, exist_ok=True)

with open(RESULTS_JSON_DIR / "faster_rcnn_metrics.json", "w", encoding="utf-8") as f:
    json.dump(frcnn_results, f, ensure_ascii=False, indent=4)

print(json.dumps(frcnn_results, ensure_ascii=False, indent=4))
print("\nМетрики сохранены:", RESULTS_JSON_DIR / "faster_rcnn_metrics.json")

In [ ]:
model_size_mb_frcnn = FRCNN_WEIGHTS_PATH.stat().st_size / (1024 * 1024)

print(f"Размер модели Faster R-CNN: {model_size_mb_frcnn:.2f} MB")

## 13. Итоговое сравнение архитектур

In [ ]:
import json
import pandas as pd
from pathlib import Path

PROJECT_DIR = Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd()

RESULTS_JSON_DIR = PROJECT_DIR / "results" / "json"
RESULTS_EXCEL_DIR = PROJECT_DIR / "results" / "excel"
RESULTS_EXCEL_DIR.mkdir(parents=True, exist_ok=True)

metrics_files = [
    RESULTS_JSON_DIR / "yolov8n_metrics.json",
    RESULTS_JSON_DIR / "yolo11n_metrics.json",
    RESULTS_JSON_DIR / "rtdetr_metrics.json",
    RESULTS_JSON_DIR / "yolov10n_metrics.json",
    RESULTS_JSON_DIR / "faster_rcnn_metrics.json",
]

model_sizes = {
    "YOLOv8n": PROJECT_DIR / "models" / "yolov8n_text_detection-2" / "weights" / "best.pt",
    "YOLO11n": PROJECT_DIR / "models" / "yolo11n_text_detection" / "weights" / "best.pt",
    "RT-DETR-L": PROJECT_DIR / "models" / "rtdetr_text_detection" / "weights" / "best.pt",
    "YOLOv10n": PROJECT_DIR / "models" / "yolov10n_text_detection" / "weights" / "best.pt",
    "Faster R-CNN ResNet50-FPN": PROJECT_DIR / "models" / "faster_rcnn_text_detection" / "faster_rcnn_text_detection.pth",
}

all_results = []

for file_path in metrics_files:
    if not file_path.exists():
        print("Файл не найден:", file_path)
        continue
    
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    
    model_name = data["model"]
    size_path = model_sizes.get(model_name)
    
    if size_path and size_path.exists():
        data["model_size_mb"] = round(size_path.stat().st_size / (1024 * 1024), 2)
    else:
        data["model_size_mb"] = None
    
    all_results.append(data)

df_comparison = pd.DataFrame(all_results)

columns_order = [
    "model",
    "task",
    "training_scheme",
    "epochs",
    "imgsz",
    "mAP50",
    "mAP50_95",
    "precision",
    "recall",
    "speed_ms_inference",
    "model_size_mb"
]

df_comparison = df_comparison[[col for col in columns_order if col in df_comparison.columns]]

csv_path = RESULTS_EXCEL_DIR / "architecture_comparison.csv"
xlsx_path = RESULTS_EXCEL_DIR / "architecture_comparison.xlsx"
json_path = RESULTS_JSON_DIR / "architecture_comparison.json"

df_comparison.to_csv(csv_path, index=False, encoding="utf-8-sig")
df_comparison.to_excel(xlsx_path, index=False)

with open(json_path, "w", encoding="utf-8") as f:
    json.dump(all_results, f, ensure_ascii=False, indent=4)

print("Итоговая таблица сравнения архитектур:")
display(df_comparison)

print("\nФайлы сохранены:")
print(csv_path)
print(xlsx_path)
print(json_path)

## 14. Демонстрационный OCR-модуль: YOLO11n + EasyOCR

In [ ]:
from pathlib import Path
import cv2
import json
import random
import torch
import numpy as np
import pandas as pd
import easyocr
from ultralytics import YOLO
import matplotlib.pyplot as plt

PROJECT_DIR = Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd()

YOLO11_WEIGHTS = PROJECT_DIR / "models" / "yolo11n_text_detection" / "weights" / "best.pt"
DATASET_DIR = PROJECT_DIR / "data" / "raw" / "coco_text_v2" / "archive"
VAL_TXT = DATASET_DIR / "val.txt"

DEMO_DIR = PROJECT_DIR / "results" / "demo_ocr"
DEMO_DIR.mkdir(parents=True, exist_ok=True)

print("YOLO11 weights:", YOLO11_WEIGHTS)
print("weights exists:", YOLO11_WEIGHTS.exists())
print("VAL exists:", VAL_TXT.exists())
print("Demo dir:", DEMO_DIR)

detector = YOLO(str(YOLO11_WEIGHTS))

reader = easyocr.Reader(
    ["en"],
    gpu=torch.cuda.is_available()
)

print("YOLO11n и EasyOCR загружены")

### 14.1. Функция детекции и распознавания текста

In [ ]:
def run_ocr_demo_on_image(
    image_path,
    detector,
    reader,
    output_dir,
    conf_threshold=0.25,
    imgsz=640
):
    image_path = Path(image_path)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    image_bgr = cv2.imread(str(image_path))
    if image_bgr is None:
        raise ValueError(f"Не удалось открыть изображение: {image_path}")

    original = image_bgr.copy()
    h, w = image_bgr.shape[:2]

    # Детекция текстовых областей
    results = detector(
        str(image_path),
        conf=conf_threshold,
        imgsz=imgsz,
        device=0 if torch.cuda.is_available() else "cpu",
        verbose=False
    )[0]

    detections = []

    if results.boxes is not None:
        boxes = results.boxes.xyxy.cpu().numpy()
        scores = results.boxes.conf.cpu().numpy()

        for i, (box, score) in enumerate(zip(boxes, scores)):
            x1, y1, x2, y2 = box.astype(int)

            # Ограничиваем координаты границами изображения
            x1 = max(0, min(x1, w - 1))
            y1 = max(0, min(y1, h - 1))
            x2 = max(0, min(x2, w - 1))
            y2 = max(0, min(y2, h - 1))

            if x2 <= x1 or y2 <= y1:
                continue

            crop = original[y1:y2, x1:x2]

            # Очень маленькие области пропускаем
            if crop.shape[0] < 5 or crop.shape[1] < 5:
                continue

            # EasyOCR ожидает RGB
            crop_rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)

            ocr_result = reader.readtext(crop_rgb, detail=1, paragraph=False)

            recognized_text = ""
            ocr_confidence = 0.0

            if len(ocr_result) > 0:
                # Берём результат с максимальной уверенностью
                best = max(ocr_result, key=lambda x: x[2])
                recognized_text = best[1]
                ocr_confidence = float(best[2])

            detections.append({
                "image": image_path.name,
                "bbox": [int(x1), int(y1), int(x2), int(y2)],
                "det_confidence": float(score),
                "recognized_text": recognized_text,
                "ocr_confidence": ocr_confidence
            })

            # Рисуем рамку и подпись
            label = recognized_text if recognized_text else "text"
            label = label[:30]

            cv2.rectangle(image_bgr, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(
                image_bgr,
                f"{label} {score:.2f}",
                (x1, max(y1 - 5, 15)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.5,
                (0, 255, 0),
                2
            )

    output_image_path = output_dir / f"ocr_result_{image_path.stem}.jpg"
    cv2.imwrite(str(output_image_path), image_bgr)

    return {
        "image": str(image_path),
        "output_image": str(output_image_path),
        "num_detected_text_regions": len(detections),
        "detections": detections
    }

### 14.2. Запуск OCR-демо на 5 изображениях

In [ ]:
# Загружаем изображения из val.txt
with open(VAL_TXT, "r", encoding="utf-8") as f:
    val_images = [Path(line.strip()) for line in f.read().splitlines() if line.strip()]

print("Всего val изображений:", len(val_images))

# Берём 5 случайных изображений для демонстрации
random.seed(42)
demo_images = random.sample(val_images, 5)

demo_results = []

for img_path in demo_images:
    result = run_ocr_demo_on_image(
        image_path=img_path,
        detector=detector,
        reader=reader,
        output_dir=DEMO_DIR,
        conf_threshold=0.25,
        imgsz=640
    )
    demo_results.append(result)
    print(f"{Path(img_path).name}: найдено областей текста — {result['num_detected_text_regions']}")

# Сохраняем JSON
demo_json_path = DEMO_DIR / "demo_ocr_results.json"

with open(demo_json_path, "w", encoding="utf-8") as f:
    json.dump(demo_results, f, ensure_ascii=False, indent=4)

# Сохраняем Excel-таблицу по найденным областям
rows = []
for item in demo_results:
    for det in item["detections"]:
        rows.append({
            "image": det["image"],
            "bbox": det["bbox"],
            "det_confidence": det["det_confidence"],
            "recognized_text": det["recognized_text"],
            "ocr_confidence": det["ocr_confidence"]
        })

demo_df = pd.DataFrame(rows)
demo_excel_path = DEMO_DIR / "demo_ocr_results.xlsx"
demo_df.to_excel(demo_excel_path, index=False)

print("\nJSON сохранён:", demo_json_path)
print("Excel сохранён:", demo_excel_path)
print("Изображения сохранены в:", DEMO_DIR)

display(demo_df.head(20))

### 14.3. Визуализация результатов OCR-демо

In [ ]:
output_images = list(DEMO_DIR.glob("ocr_result_*.jpg"))

for output_path in output_images[:5]:
    img = cv2.imread(str(output_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    plt.figure(figsize=(12, 8))
    plt.imshow(img)
    plt.title(output_path.name)
    plt.axis("off")
    plt.show()

## 15. Расширенный запуск OCR и подбор примеров

In [ ]:
# Расширенный запуск OCR-демо для подбора удачных и ошибочных примеров

EXTENDED_DEMO_DIR = PROJECT_DIR / "results" / "demo_ocr_extended"
EXTENDED_DEMO_DIR.mkdir(parents=True, exist_ok=True)

with open(VAL_TXT, "r", encoding="utf-8") as f:
    val_images = [Path(line.strip()) for line in f.read().splitlines() if line.strip()]

print("Всего val изображений:", len(val_images))

random.seed(123)
extended_demo_images = random.sample(val_images, 30)

extended_demo_results = []

for img_path in extended_demo_images:
    result = run_ocr_demo_on_image(
        image_path=img_path,
        detector=detector,
        reader=reader,
        output_dir=EXTENDED_DEMO_DIR,
        conf_threshold=0.25,
        imgsz=640
    )
    extended_demo_results.append(result)
    print(f"{Path(img_path).name}: найдено областей текста — {result['num_detected_text_regions']}")

# Сохраняем JSON
extended_json_path = EXTENDED_DEMO_DIR / "extended_demo_ocr_results.json"

with open(extended_json_path, "w", encoding="utf-8") as f:
    json.dump(extended_demo_results, f, ensure_ascii=False, indent=4)

# Сохраняем Excel
rows = []

for item in extended_demo_results:
    for det in item["detections"]:
        rows.append({
            "image": det["image"],
            "bbox": det["bbox"],
            "det_confidence": det["det_confidence"],
            "recognized_text": det["recognized_text"],
            "ocr_confidence": det["ocr_confidence"]
        })

extended_demo_df = pd.DataFrame(rows)
extended_excel_path = EXTENDED_DEMO_DIR / "extended_demo_ocr_results.xlsx"
extended_demo_df.to_excel(extended_excel_path, index=False)

print("\nJSON сохранён:", extended_json_path)
print("Excel сохранён:", extended_excel_path)
print("Изображения сохранены в:", EXTENDED_DEMO_DIR)

display(extended_demo_df.head(30))

### 15.1. Визуализация расширенного OCR-демо

In [ ]:
output_images = list(EXTENDED_DEMO_DIR.glob("ocr_result_*.jpg"))

print("Всего сохранённых изображений:", len(output_images))

for output_path in output_images[:10]:
    img = cv2.imread(str(output_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    plt.figure(figsize=(12, 8))
    plt.imshow(img)
    plt.title(output_path.name)
    plt.axis("off")
    plt.show()

### 15.2. Автоматический отбор удачных и ошибочных примеров

In [ ]:
from pathlib import Path
import pandas as pd
import shutil
import json

PROJECT_DIR = Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd()

EXTENDED_DEMO_DIR = PROJECT_DIR / "results" / "demo_ocr_extended"
EXTENDED_EXCEL_PATH = EXTENDED_DEMO_DIR / "extended_demo_ocr_results.xlsx"

SUCCESS_DIR = PROJECT_DIR / "report_materials" / "successful_examples"
ERROR_DIR = PROJECT_DIR / "report_materials" / "error_examples"

SUCCESS_DIR.mkdir(parents=True, exist_ok=True)
ERROR_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_excel(EXTENDED_EXCEL_PATH)

print("Всего найденных текстовых областей:", len(df))

# Кандидаты на удачные примеры:
# есть распознанный текст и высокая уверенность OCR
success_candidates = df[
    (df["recognized_text"].notna()) &
    (df["recognized_text"].astype(str).str.strip() != "") &
    (df["ocr_confidence"] >= 0.5)
].copy()

# Кандидаты на ошибочные примеры:
# текстовая область найдена, но OCR пустой или уверенность низкая
error_candidates = df[
    (df["recognized_text"].isna()) |
    (df["recognized_text"].astype(str).str.strip() == "") |
    (df["ocr_confidence"] < 0.1)
].copy()

print("Кандидатов на удачные примеры:", len(success_candidates))
print("Кандидатов на ошибочные примеры:", len(error_candidates))

# Берём уникальные изображения, чтобы не копировать одно и то же много раз
success_images = success_candidates["image"].drop_duplicates().head(5).tolist()
error_images = error_candidates["image"].drop_duplicates().head(5).tolist()

def copy_demo_images(image_names, target_dir, prefix):
    copied = []
    
    for i, image_name in enumerate(image_names, start=1):
        stem = Path(image_name).stem
        src = EXTENDED_DEMO_DIR / f"ocr_result_{stem}.jpg"
        
        if src.exists():
            dst = target_dir / f"{prefix}_{i}_{src.name}"
            shutil.copy2(src, dst)
            copied.append(str(dst))
        else:
            print("Не найден файл:", src)
    
    return copied

copied_success = copy_demo_images(success_images, SUCCESS_DIR, "success")
copied_errors = copy_demo_images(error_images, ERROR_DIR, "error")

selection_info = {
    "success_candidates_count": int(len(success_candidates)),
    "error_candidates_count": int(len(error_candidates)),
    "copied_success_examples": copied_success,
    "copied_error_examples": copied_errors
}

selection_json_path = PROJECT_DIR / "results" / "json" / "selected_demo_examples.json"

with open(selection_json_path, "w", encoding="utf-8") as f:
    json.dump(selection_info, f, ensure_ascii=False, indent=4)

print("\nСкопированы удачные примеры:")
for path in copied_success:
    print(path)

print("\nСкопированы ошибочные примеры:")
for path in copied_errors:
    print(path)

print("\nJSON сохранён:", selection_json_path)

## 16. Создание отдельного демонстрационного модуля demo_ocr.py

In [ ]:
from pathlib import Path

PROJECT_DIR = Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd()
DEMO_SCRIPT_PATH = PROJECT_DIR / "demo" / "demo_ocr.py"
DEMO_SCRIPT_PATH.parent.mkdir(parents=True, exist_ok=True)

demo_code = r'''
import argparse
import json
import random
from pathlib import Path

import cv2
import pandas as pd
import torch
import easyocr
from ultralytics import YOLO


def run_ocr_on_image(image_path, detector, reader, output_dir, conf_threshold=0.25, imgsz=640):
    image_path = Path(image_path)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    image_bgr = cv2.imread(str(image_path))
    if image_bgr is None:
        print(f"Не удалось открыть изображение: {image_path}")
        return None

    original = image_bgr.copy()
    h, w = image_bgr.shape[:2]

    results = detector(
        str(image_path),
        conf=conf_threshold,
        imgsz=imgsz,
        device=0 if torch.cuda.is_available() else "cpu",
        verbose=False
    )[0]

    detections = []

    if results.boxes is not None:
        boxes = results.boxes.xyxy.cpu().numpy()
        scores = results.boxes.conf.cpu().numpy()

        for box, score in zip(boxes, scores):
            x1, y1, x2, y2 = box.astype(int)

            x1 = max(0, min(x1, w - 1))
            y1 = max(0, min(y1, h - 1))
            x2 = max(0, min(x2, w - 1))
            y2 = max(0, min(y2, h - 1))

            if x2 <= x1 or y2 <= y1:
                continue

            crop = original[y1:y2, x1:x2]

            if crop.shape[0] < 5 or crop.shape[1] < 5:
                continue

            crop_rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
            ocr_result = reader.readtext(crop_rgb, detail=1, paragraph=False)

            recognized_text = ""
            ocr_confidence = 0.0

            if len(ocr_result) > 0:
                best = max(ocr_result, key=lambda x: x[2])
                recognized_text = best[1]
                ocr_confidence = float(best[2])

            detections.append({
                "image": image_path.name,
                "bbox": [int(x1), int(y1), int(x2), int(y2)],
                "det_confidence": float(score),
                "recognized_text": recognized_text,
                "ocr_confidence": ocr_confidence
            })

            label = recognized_text if recognized_text else "text"
            label = label[:30]

            cv2.rectangle(image_bgr, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(
                image_bgr,
                f"{label} {score:.2f}",
                (x1, max(y1 - 5, 15)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.5,
                (0, 255, 0),
                2
            )

    output_image_path = output_dir / f"ocr_result_{image_path.stem}.jpg"
    cv2.imwrite(str(output_image_path), image_bgr)

    return {
        "image": str(image_path),
        "output_image": str(output_image_path),
        "num_detected_text_regions": len(detections),
        "detections": detections
    }


def collect_input_images(input_path, default_val_txt, num_samples):
    input_path = Path(input_path) if input_path else None

    if input_path and input_path.is_file():
        return [input_path]

    if input_path and input_path.is_dir():
        images = []
        for ext in ["*.jpg", "*.jpeg", "*.png", "*.bmp"]:
            images.extend(input_path.glob(ext))
        return images

    with open(default_val_txt, "r", encoding="utf-8") as f:
        val_images = [Path(line.strip()) for line in f.read().splitlines() if line.strip()]

    random.seed(42)
    return random.sample(val_images, min(num_samples, len(val_images)))


def main():
    parser = argparse.ArgumentParser(description="Demo OCR module: text detection + recognition")
    parser.add_argument("--input", type=str, default=None, help="Path to image or folder with images")
    parser.add_argument("--output", type=str, default=None, help="Output folder")
    parser.add_argument("--conf", type=float, default=0.25, help="Detection confidence threshold")
    parser.add_argument("--num_samples", type=int, default=5, help="Number of validation images if input is not specified")
    args = parser.parse_args()

    project_dir = Path(__file__).resolve().parents[1]

    weights_path = project_dir / "models" / "yolo11n_text_detection" / "weights" / "best.pt"
    val_txt = project_dir / "data" / "raw" / "coco_text_v2" / "archive" / "val.txt"

    output_dir = Path(args.output) if args.output else project_dir / "results" / "demo_ocr_script"
    output_dir.mkdir(parents=True, exist_ok=True)

    print("Project dir:", project_dir)
    print("Weights:", weights_path)
    print("Weights exists:", weights_path.exists())
    print("Output dir:", output_dir)

    detector = YOLO(str(weights_path))

    reader = easyocr.Reader(
        ["en"],
        gpu=torch.cuda.is_available()
    )

    input_images = collect_input_images(args.input, val_txt, args.num_samples)
    print("Images for processing:", len(input_images))

    all_results = []

    for image_path in input_images:
        result = run_ocr_on_image(
            image_path=image_path,
            detector=detector,
            reader=reader,
            output_dir=output_dir,
            conf_threshold=args.conf,
            imgsz=640
        )

        if result is not None:
            all_results.append(result)
            print(f"{Path(image_path).name}: text regions = {result['num_detected_text_regions']}")

    json_path = output_dir / "demo_ocr_script_results.json"

    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(all_results, f, ensure_ascii=False, indent=4)

    rows = []

    for item in all_results:
        for det in item["detections"]:
            rows.append({
                "image": det["image"],
                "bbox": det["bbox"],
                "det_confidence": det["det_confidence"],
                "recognized_text": det["recognized_text"],
                "ocr_confidence": det["ocr_confidence"]
            })

    excel_path = output_dir / "demo_ocr_script_results.xlsx"
    pd.DataFrame(rows).to_excel(excel_path, index=False)

    print("JSON saved:", json_path)
    print("Excel saved:", excel_path)
    print("Annotated images saved:", output_dir)


if __name__ == "__main__":
    main()
'''

DEMO_SCRIPT_PATH.write_text(demo_code, encoding="utf-8")

print("Демо-модуль создан:", DEMO_SCRIPT_PATH)
print("Файл существует:", DEMO_SCRIPT_PATH.exists())

### 16.1. Проверка запуска demo_ocr.py

In [ ]:
import sys
from pathlib import Path

PROJECT_DIR = Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd()
DEMO_SCRIPT_PATH = PROJECT_DIR / "demo" / "demo_ocr.py"

!{sys.executable} "{DEMO_SCRIPT_PATH}" --num_samples 5 --conf 0.25

## 17. Финальная проверка файлов проекта

In [ ]:
from pathlib import Path
import json

PROJECT_DIR = Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd()

required_paths = {
    "YAML конфиг": PROJECT_DIR / "ocr_text_detection.yaml",

    "YOLOv8n best.pt": PROJECT_DIR / "models" / "yolov8n_text_detection-2" / "weights" / "best.pt",
    "YOLO11n best.pt": PROJECT_DIR / "models" / "yolo11n_text_detection" / "weights" / "best.pt",
    "RT-DETR best.pt": PROJECT_DIR / "models" / "rtdetr_text_detection" / "weights" / "best.pt",
    "YOLOv10n best.pt": PROJECT_DIR / "models" / "yolov10n_text_detection" / "weights" / "best.pt",
    "Faster R-CNN model": PROJECT_DIR / "models" / "faster_rcnn_text_detection" / "faster_rcnn_text_detection.pth",

    "YOLOv8n metrics": PROJECT_DIR / "results" / "json" / "yolov8n_metrics.json",
    "YOLO11n metrics": PROJECT_DIR / "results" / "json" / "yolo11n_metrics.json",
    "RT-DETR metrics": PROJECT_DIR / "results" / "json" / "rtdetr_metrics.json",
    "YOLOv10n metrics": PROJECT_DIR / "results" / "json" / "yolov10n_metrics.json",
    "Faster R-CNN metrics": PROJECT_DIR / "results" / "json" / "faster_rcnn_metrics.json",

    "Architecture comparison JSON": PROJECT_DIR / "results" / "json" / "architecture_comparison.json",
    "Architecture comparison Excel": PROJECT_DIR / "results" / "excel" / "architecture_comparison.xlsx",

    "Demo script": PROJECT_DIR / "demo" / "demo_ocr.py",
    "Demo OCR JSON": PROJECT_DIR / "results" / "demo_ocr_script" / "demo_ocr_script_results.json",
    "Demo OCR Excel": PROJECT_DIR / "results" / "demo_ocr_script" / "demo_ocr_script_results.xlsx",

    "Extended OCR JSON": PROJECT_DIR / "results" / "demo_ocr_extended" / "extended_demo_ocr_results.json",
    "Extended OCR Excel": PROJECT_DIR / "results" / "demo_ocr_extended" / "extended_demo_ocr_results.xlsx",

    "Successful examples folder": PROJECT_DIR / "report_materials" / "successful_examples",
    "Error examples folder": PROJECT_DIR / "report_materials" / "error_examples",
}

print("Проверка файлов проекта:\n")

all_ok = True

for name, path in required_paths.items():
    exists = path.exists()
    status = "OK" if exists else "НЕ НАЙДЕНО"
    print(f"{status:10} | {name}: {path}")
    
    if not exists:
        all_ok = False

print("\nИтог:")
if all_ok:
    print("Все основные файлы проекта на месте.")
else:
    print("Есть отсутствующие файлы. Нужно проверить строки с НЕ НАЙДЕНО.")

### 17.1. Проверка количества сохранённых демонстрационных изображений

In [ ]:
from pathlib import Path

PROJECT_DIR = Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd()

demo_script_dir = PROJECT_DIR / "results" / "demo_ocr_script"
extended_demo_dir = PROJECT_DIR / "results" / "demo_ocr_extended"
success_dir = PROJECT_DIR / "report_materials" / "successful_examples"
error_dir = PROJECT_DIR / "report_materials" / "error_examples"

print("Изображений demo_ocr_script:", len(list(demo_script_dir.glob("ocr_result_*.jpg"))))
print("Изображений demo_ocr_extended:", len(list(extended_demo_dir.glob("ocr_result_*.jpg"))))
print("Удачных примеров для отчёта:", len(list(success_dir.glob("*.jpg"))))
print("Ошибочных примеров для отчёта:", len(list(error_dir.glob("*.jpg"))))

### 15.3. Дополнительный поиск удачных примеров OCR

In [ ]:
from pathlib import Path
import random
import json
import cv2
import torch
import pandas as pd
import matplotlib.pyplot as plt
from ultralytics import YOLO
import easyocr

PROJECT_DIR = Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd()

VAL_TXT = PROJECT_DIR / "data" / "raw" / "coco_text_v2" / "archive" / "val.txt"
WEIGHTS_PATH = PROJECT_DIR / "models" / "yolo11n_text_detection" / "weights" / "best.pt"

SUCCESS_SEARCH_DIR = PROJECT_DIR / "results" / "success_search_improved"
SUCCESS_SEARCH_DIR.mkdir(parents=True, exist_ok=True)

detector = YOLO(str(WEIGHTS_PATH))
reader = easyocr.Reader(["en"], gpu=torch.cuda.is_available())

with open(VAL_TXT, "r", encoding="utf-8") as f:
    val_images = [Path(line.strip()) for line in f.read().splitlines() if line.strip()]

NUM_IMAGES = 250

random.seed(2026)
sample_images = random.sample(val_images, min(NUM_IMAGES, len(val_images)))

print("Будет проверено изображений:", len(sample_images))
print("Результаты будут сохранены в:", SUCCESS_SEARCH_DIR)

In [ ]:
def prepare_crop_for_ocr(crop):
    h, w = crop.shape[:2]
    
    if h == 0 or w == 0:
        return crop
    
    # Увеличиваем маленькие текстовые области
    if max(h, w) < 80:
        scale = 4
    elif max(h, w) < 160:
        scale = 3
    else:
        scale = 2
    
    crop = cv2.resize(
        crop,
        None,
        fx=scale,
        fy=scale,
        interpolation=cv2.INTER_CUBIC
    )
    
    return crop


all_rows = []
saved_candidate_images = []

for idx, image_path in enumerate(sample_images, start=1):
    image_bgr = cv2.imread(str(image_path))
    
    if image_bgr is None:
        continue
    
    annotated = image_bgr.copy()
    h, w = image_bgr.shape[:2]
    
    result = detector(
        str(image_path),
        conf=0.25,
        imgsz=640,
        device=0 if torch.cuda.is_available() else "cpu",
        verbose=False
    )[0]
    
    good_detections_on_image = 0
    
    if result.boxes is not None:
        boxes = result.boxes.xyxy.cpu().numpy()
        scores = result.boxes.conf.cpu().numpy()
        
        for box, det_conf in zip(boxes, scores):
            x1, y1, x2, y2 = box.astype(int)
            
            x1 = max(0, min(x1, w - 1))
            y1 = max(0, min(y1, h - 1))
            x2 = max(0, min(x2, w - 1))
            y2 = max(0, min(y2, h - 1))
            
            if x2 <= x1 or y2 <= y1:
                continue
            
            # Добавляем отступ вокруг текстовой области
            box_w = x2 - x1
            box_h = y2 - y1
            pad = int(0.25 * max(box_w, box_h))
            
            px1 = max(0, x1 - pad)
            py1 = max(0, y1 - pad)
            px2 = min(w - 1, x2 + pad)
            py2 = min(h - 1, y2 + pad)
            
            crop = image_bgr[py1:py2, px1:px2]
            
            if crop.shape[0] < 5 or crop.shape[1] < 5:
                continue
            
            crop = prepare_crop_for_ocr(crop)
            crop_rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
            
            ocr_result = reader.readtext(
                crop_rgb,
                detail=1,
                paragraph=False,
                decoder="beamsearch"
            )
            
            recognized_text = ""
            ocr_confidence = 0.0
            
            if len(ocr_result) > 0:
                best = max(ocr_result, key=lambda x: x[2])
                recognized_text = best[1]
                ocr_confidence = float(best[2])
            
            row = {
                "image": image_path.name,
                "bbox": [int(x1), int(y1), int(x2), int(y2)],
                "det_confidence": float(det_conf),
                "recognized_text": recognized_text,
                "ocr_confidence": float(ocr_confidence)
            }
            
            all_rows.append(row)
            
            # Кандидат в удачные: есть непустой текст, не слишком короткий и OCR уверен
            is_good_candidate = (
                recognized_text.strip() != "" and
                len(recognized_text.strip()) >= 2 and
                ocr_confidence >= 0.65
            )
            
            if is_good_candidate:
                good_detections_on_image += 1
                label = recognized_text[:25]
                
                cv2.rectangle(annotated, (x1, y1), (x2, y2), (0, 255, 0), 2)
                cv2.putText(
                    annotated,
                    f"{label} {ocr_confidence:.2f}",
                    (x1, max(y1 - 5, 15)),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.5,
                    (0, 255, 0),
                    2
                )
    
    if good_detections_on_image > 0:
        output_path = SUCCESS_SEARCH_DIR / f"candidate_{image_path.stem}.jpg"
        cv2.imwrite(str(output_path), annotated)
        saved_candidate_images.append(output_path)
    
    if idx % 25 == 0:
        print(f"Проверено изображений: {idx}/{len(sample_images)}")

df_success_search = pd.DataFrame(all_rows)

excel_path = SUCCESS_SEARCH_DIR / "success_search_improved.xlsx"
json_path = SUCCESS_SEARCH_DIR / "success_search_improved.json"

df_success_search.to_excel(excel_path, index=False)

with open(json_path, "w", encoding="utf-8") as f:
    json.dump(all_rows, f, ensure_ascii=False, indent=4)

print("\nГотово.")
print("Всего OCR-строк:", len(df_success_search))
print("Сохранено изображений-кандидатов:", len(saved_candidate_images))
print("Excel:", excel_path)
print("JSON:", json_path)

### 15.4. Визуальный просмотр найденных кандидатов

In [ ]:
candidate_images = sorted(SUCCESS_SEARCH_DIR.glob("candidate_*.jpg"))

print("Всего кандидатов:", len(candidate_images))

for image_path in candidate_images[:20]:
    img = cv2.imread(str(image_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    plt.figure(figsize=(10, 7))
    plt.imshow(img)
    plt.title(image_path.name)
    plt.axis("off")
    plt.show()